# Perbaikan SENA CPU v0.5

Eksperimen baru dari bobot baseline SENA v1 yang tersedia, bukan reproduksi bobot kandidat Colab yang belum ditemukan. Menggunakan development split baru dari 730 label manusia yang sudah diketahui dan 64 pasangan latihan berlabel asisten. Validation lama 620 dipensiunkan sebagai untouched validation. Pilot 50 tidak dipakai untuk training atau pemilihan checkpoint.

BERT dibekukan kecuali dua layer akhir, pooler dan classifier. Konfigurasi, kriteria seleksi dan penghentian dini dibekukan sebelum run. Hasil development dan diagnostik yang telah dilihat tidak dinyatakan sebagai test penerimaan independen. Target mutu tidak diturunkan.


In [ ]:
from pathlib import Path
import os,sys,json,subprocess,hashlib
ROOT = Path(os.environ['SLR09_PACKAGE_ROOT'])
ASSETS = Path(os.environ['SLR09_ASSET_DIR'])
OUTPUT = Path(os.environ['SLR09_TRAIN_OUTPUT'])
print('Python:',sys.version)
print('Bertolak dari baseline SENA v1; tidak memakai pilot50 untuk fit/seleksi.')
print('Source SHA:',hashlib.sha256((ROOT/'code/quality_fix/train_sena_cpu.py').read_bytes()).hexdigest())


In [ ]:
command=[sys.executable,str(ROOT/'code/quality_fix/train_sena_cpu.py'),
 '--root',str(ROOT),'--archive',str(ASSETS/'SILAR_SENA_v1.zip'),'--output',str(OUTPUT)]
completed=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in completed.stdout:
 print(line,end='',flush=True)
completed.wait()
if completed.returncode:raise RuntimeError('Training tidak selesai; periksa run_state dan log, jangan klaim lulus.')


In [ ]:
state=json.loads((OUTPUT/'run_state.json').read_text())
metrics=json.loads((OUTPUT/'best_development_metrics.json').read_text())
print(json.dumps(state,indent=2))
print(json.dumps(metrics,indent=2))
assert state['training_executed'] and state['status']=='SUCCEEDED'
print('Checkpoint dipilih pada development validation. Penerimaan mutu akhir belum diberikan.')
